In [1]:
# ── CELL 1: Install Requirements ──────────────────────────────────────────
%pip install gradio ultralytics tensorflow opencv-python-headless Pillow numpy matplotlib pyyaml
print("✅ All packages installed!")

Note: you may need to restart the kernel to use updated packages.
✅ All packages installed!


In [16]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────
import gradio as gr
import cv2
import numpy as np
import os
import yaml
import csv
import tempfile
import shutil
from datetime import datetime
from pathlib import Path
from collections import deque
import pandas as pd
import tensorflow as tf
from ultralytics import YOLO
from PIL import Image
import torch

print("✅ All imports done!")

✅ All imports done!


In [17]:
# ── CELL 3: Load Models ───────────────────────────────────────────────────
BASE_DIR   = r"C:\Users\ASUS\indian_traffic_model"
YOLO_PATH  = r"C:\Users\ASUS\indian_traffic_model\task3_phase2\weights\best.pt"
ACC_PATH   = r"C:\Users\ASUS\indian_traffic_model\accident_model\accident_model.h5"
DATA_YAML  = r"C:\Users\ASUS\task3.yaml"

print("⏳ Loading models...")

# ── YOLO — single model handles ALL 75 classes ────────────────────────────
# vehicles (0-8) + signs (9-72) + helmet (73-74)
yolo_model = YOLO(YOLO_PATH)
print(f"✅ YOLO loaded — {len(yolo_model.names)} classes")
print(f"   Vehicles : 0–8  | Signs : 9–72  | Helmet : 73–74")

# ── Accident CNN ───────────────────────────────────────────────────────────
accident_model  = tf.keras.models.load_model(ACC_PATH)
IMG_H           = accident_model.input_shape[1]
IMG_W           = accident_model.input_shape[2]
output_units    = accident_model.output_shape[-1]
ACCIDENT_LABELS = ['no_accident', 'accident']
print(f"✅ Accident CNN loaded — input {IMG_H}x{IMG_W}")

# ── GPU check ──────────────────────────────────────────────────────────────
print(f"\n🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

# ── Class ID buckets (from single YOLO model) ─────────────────────────────
VEHICLE_IDS = set(range(9))       # 0–8
SIGN_IDS    = set(range(9, 73))   # 9–72
HELMET_IDS  = {73, 74}            # 73=helmet, 74=no_helmet

print("\n📋 Pipeline Ready:")
print(f"   Single YOLO model → vehicles + signs + helmet (75 classes)")
print(f"   Accident CNN      → binary classifier (accident / no_accident)")


⏳ Loading models...
✅ YOLO loaded — 75 classes
   Vehicles : 0–8  | Signs : 9–72  | Helmet : 73–74


✅ Accident CNN loaded — input 250x250

🖥️  GPU: NVIDIA GeForce RTX 2050

📋 Pipeline Ready:
   Single YOLO model → vehicles + signs + helmet (75 classes)
   Accident CNN      → binary classifier (accident / no_accident)


In [19]:
# ── CELL 4: Global Counters + Log Paths ───────────────────────────────────
vehicle_counter          = 0
helmet_violation_counter = 0
traffic_sign_counter     = 0
accident_counter         = 0

ACCIDENT_HISTORY   = deque(maxlen=5)
ACCIDENT_THRESHOLD = 0.80
MIN_VEHICLES       = 2

VIOLATION_LOG = os.path.join(os.getcwd(), 'violation_logs.csv')
ACCIDENT_LOG  = os.path.join(os.getcwd(), 'accident_logs.csv')

print("✅ Counters + log paths ready!")

✅ Counters + log paths ready!


In [20]:
# ── CELL 5: Helper Functions ──────────────────────────────────────────────

# ── Traffic Density ────────────────────────────────────────────────────────
def traffic_density(count):
    if count < 5:   return "🟢 Low Traffic"
    elif count < 15: return "🟡 Medium Traffic"
    else:            return "🔴 Heavy Traffic"

# ── Helmet Violation (class name based) ───────────────────────────────────
def helmet_violation(classes):
    if ('motorcycle' in classes or 'bike' in classes) and 'no_helmet' in classes:
        return True
    return False

# ── Overlap check (helmet box vs vehicle box) ──────────────────────────────
def check_overlap(box1, box2):
    x1,y1,x2,y2   = box1
    x1b,y1b,x2b,y2b = box2
    if x1 > x2b or x2 < x1b: return False
    if y1 > y2b or y2 < y1b: return False
    return True

# ── Parse YOLO detections ──────────────────────────────────────────────────
def parse_detections(results):
    vehicles, signs, helmets = [], [], []
    for box in results[0].boxes:
        cls_id = int(box.cls)
        name   = yolo_model.names[cls_id]
        conf   = float(box.conf)
        if   cls_id in VEHICLE_IDS: 
            vehicles.append((name, conf, box.xyxy[0].cpu().numpy()))
        elif cls_id in SIGN_IDS:    
            signs.append((name, conf))
        elif cls_id in HELMET_IDS:  
            helmets.append((name, conf))
    return vehicles, signs, helmets

# ── Accident CNN ───────────────────────────────────────────────────────────
def get_dynamic_threshold(vehicle_count):
    if vehicle_count >= 5:   return 0.70
    elif vehicle_count >= 3: return 0.75
    else:                    return 0.80

def predict_accident(frame):
    resized    = cv2.resize(frame, (IMG_W, IMG_H))
    rgb        = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    normalized = rgb.astype(np.float32) / 255.0
    batch      = np.expand_dims(normalized, axis=0)
    pred       = accident_model.predict(batch, verbose=0)
    if output_units == 1:
        conf   = float(pred[0][0])
        label  = 'accident' if conf >= ACCIDENT_THRESHOLD else 'no_accident'
    else:
        idx    = int(np.argmax(pred[0]))
        conf   = float(pred[0][idx])
        label  = ACCIDENT_LABELS[idx] if idx < len(ACCIDENT_LABELS) else str(idx)
    is_acc = (label == 'accident') and (conf >= ACCIDENT_THRESHOLD)
    return label, conf, is_acc

def predict_accident_smoothed(frame, yolo_results):
    boxes         = yolo_results[0].boxes
    vehicle_count = sum(1 for b in boxes if int(b.cls) in VEHICLE_IDS)
    if vehicle_count < MIN_VEHICLES:
        ACCIDENT_HISTORY.append(False)
        return 'no_accident', 0.0, False, vehicle_count
    label, conf, _ = predict_accident(frame)
    dyn_thresh     = get_dynamic_threshold(vehicle_count)
    is_acc         = label == 'accident' and conf >= dyn_thresh and vehicle_count >= 2
    ACCIDENT_HISTORY.append(is_acc)
    smoothed       = sum(ACCIDENT_HISTORY) >= 3
    return label, conf, smoothed, vehicle_count

def has_large_vehicle(yolo_results, min_area=5000):
    for box in yolo_results[0].boxes:
        if int(box.cls) in VEHICLE_IDS:
            x1,y1,x2,y2 = box.xyxy[0]
            if (x2-x1)*(y2-y1) > min_area:
                return True
    return False

def draw_accident_overlay(frame, label, conf, is_accident, vehicle_count=0):
    h, w    = frame.shape[:2]
    overlay = frame.copy()
    if is_accident:
        cv2.rectangle(overlay, (0,0), (w,90), (0,0,200), -1)
        cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
        cv2.putText(frame, f"ACCIDENT DETECTED  {conf:.0%}",
                    (15,45), cv2.FONT_HERSHEY_DUPLEX, 1.2, (255,255,255), 2)
        cv2.putText(frame, f"Vehicles in frame: {vehicle_count}",
                    (15,78), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200,200,255), 1)
    else:
        cv2.rectangle(frame, (0,0), (300,40), (0,160,0), -1)
        cv2.putText(frame, f"SAFE  {conf:.0%}",
                    (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255,255,255), 2)
    return frame

# ── CSV Logging ────────────────────────────────────────────────────────────
def log_to_csv(path, row, headers):
    file_exists = os.path.isfile(path)
    with open(path, mode='a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(headers)
        w.writerow(row)

def get_log_df(path, cols):
    if os.path.exists(path):
        return pd.read_csv(path).tail(15).iloc[::-1]
    return pd.DataFrame(columns=cols)

# ── Stats text builder ─────────────────────────────────────────────────────
def build_stats(vehicles, signs, helmets,
                acc_label, acc_conf, is_accident,
                vehicle_count, density):
    lines = [
        "=" * 42, "      🚦 DETECTION SUMMARY", "=" * 42,
        f"  🚗 Vehicles in frame : {len(vehicles)}",
        f"  🚦 Signs detected    : {len(signs)}",
        f"  🪖 Helmet detections : {len(helmets)}",
        f"  ⚠️  No-helmet alerts : {sum(1 for n,_ in helmets if n=='no_helmet')}",
        "", "─ Traffic Density ───────────────────",
        f"  {density}",
        "", "─ Session Totals ────────────────────",
        f"  🚗 Total vehicles    : {vehicle_counter}",
        f"  🚨 Total violations  : {helmet_violation_counter}",
        f"  🚦 Total signs seen  : {traffic_sign_counter}",
        f"  💥 Total accidents   : {accident_counter}",
        "", "─ Accident Status ───────────────────",
        f"  {'🚨 ACCIDENT DETECTED' if is_accident else '✅ SAFE'}  {acc_conf:.1%}",
        "=" * 42,
    ]
    if vehicles:
        lines += ["─ Vehicles ──────────────────────────"]
        for name, conf, _ in sorted(vehicles, key=lambda x:-x[1])[:6]:
            lines.append(f"  {name:<18} {conf:.0%}  {'█'*int(conf*15)}")
    if helmets:
        lines += ["─ Helmet Status ─────────────────────"]
        for name, conf in helmets[:6]:
            lines.append(f"  {'🪖' if name=='helmet' else '⚠️'} {name:<16} {conf:.0%}")
    if signs:
        lines += ["─ Traffic Signs ─────────────────────"]
        for name, conf in sorted(signs, key=lambda x:-x[1])[:6]:
            lines.append(f"  🚦 {name:<16} {conf:.0%}")
    lines.append("=" * 42)
    return "\n".join(lines)

def reset_counters():
    global vehicle_counter, helmet_violation_counter
    global traffic_sign_counter, accident_counter
    vehicle_counter = helmet_violation_counter = 0
    traffic_sign_counter = accident_counter = 0
    ACCIDENT_HISTORY.clear()
    return "✅ All counters reset to zero!"

print("✅ All helper functions ready!")

✅ All helper functions ready!


In [27]:
def detect_image(pil_image, conf_threshold, run_accident):
    global vehicle_counter, helmet_violation_counter
    global traffic_sign_counter, accident_counter

    if pil_image is None:
        return None, "⚠️ No image uploaded.", get_log_df(
            VIOLATION_LOG, ['DateTime','Type','Confidence','Source'])

    frame = cv2.cvtColor(np.array(pil_image.convert("RGB")), cv2.COLOR_RGB2BGR)

    # ── Single YOLO model — all 75 classes in one pass ────────────────────
    results   = yolo_model.predict(
        frame, conf=conf_threshold, iou=0.45, imgsz=1280, verbose=False)
    annotated = results[0].plot(line_width=2, font_size=7)

    # ── Parse all detections from single model ────────────────────────────
    vehicles, signs, helmets = parse_detections(results)

    # ── Update general counters ────────────────────────────────────────────
    vehicle_counter      += len(vehicles)
    traffic_sign_counter += len(signs)

    # ── BULLETPROOF VIOLATION ROUTING ──────────────────────────────────────
    for box in results[0].boxes:
        class_id = int(box.cls[0])
        class_name = yolo_model.names[class_id]
        conf = float(box.conf[0])

        if class_name == 'no_helmet':
            helmet_violation_counter += 1
            violation_text = "Motorcycle - No Helmet"
            
            log_to_csv(
                VIOLATION_LOG, 
                [datetime.now().strftime("%Y-%m-%d %H:%M:%S"), violation_text, f"{conf:.2f}", "Image Upload"], 
                ['DateTime', 'Type', 'Confidence', 'Source']
            )

        elif class_name == 'no_parking': # Ready for future expansion
            traffic_sign_counter += 1
            violation_text = "Illegal Parking Detected"
            
            log_to_csv(
                VIOLATION_LOG, 
                [datetime.now().strftime("%Y-%m-%d %H:%M:%S"), violation_text, f"{conf:.2f}", "Image Upload"], 
                ['DateTime', 'Type', 'Confidence', 'Source']
            )

    # ── Density ────────────────────────────────────────────────────────────
    density = traffic_density(len(vehicles))

    # ── Accident CNN ───────────────────────────────────────────────────────
    if run_accident and len(vehicles) >= 2 and has_large_vehicle(results):
        acc_label, acc_conf, is_accident, _ = predict_accident_smoothed(frame, results)
        if is_accident:
            accident_counter += 1
            log_to_csv(ACCIDENT_LOG,
                [datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                 'N/A', 0, acc_conf],
                ['DateTime','Frame','Video_Time_sec','Confidence'])
    else:
        acc_label, acc_conf, is_accident = 'no_accident', 0.0, False

    stats   = build_stats(vehicles, signs, helmets,
                          acc_label, acc_conf, is_accident,
                          len(vehicles), density)
    out_pil = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    vlog    = get_log_df(VIOLATION_LOG,
                ['DateTime','Type','Confidence','Source'])
    return out_pil, stats, vlog

print("✅ detect_image ready!")




✅ detect_image ready!


In [28]:

def detect_video(video_path, conf_threshold, run_accident, acc_every_n):
    global vehicle_counter, helmet_violation_counter
    global traffic_sign_counter, accident_counter

    if video_path is None:
        return None, "⚠️ No video uploaded.", None, None

    cap          = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25
    W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out_path = os.path.join(tempfile.gettempdir(), 'gradio_output.mp4')
    writer   = cv2.VideoWriter(
        out_path, cv2.VideoWriter_fourcc(*'avc1'), fps, (W, H))

    ACCIDENT_HISTORY.clear()
    acc_label, acc_conf, acc_flag, acc_vehicles = 'no_accident', 0.0, False, 0
    accident_events = []
    frame_idx       = 0
    total_v = total_s = total_h = no_helm = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_idx += 1

        # ── Single YOLO model — all 75 classes ────────────────────────────
        results   = yolo_model.predict(
            frame, conf=conf_threshold, iou=0.45,
            imgsz=640, verbose=False)
        annotated = results[0].plot(line_width=2, font_size=7)

        # ── Parse all detections ───────────────────────────────────────────
        vehicles, signs, helmets = parse_detections(results)

        # ── Update general counters ────────────────────────────────────────
        total_v  += len(vehicles)
        total_s  += len(signs)
        total_h  += len(helmets)

        vehicle_counter      += len(vehicles)
        traffic_sign_counter += len(signs)

        # ── BULLETPROOF VIOLATION ROUTING ──────────────────────────────────
        for box in results[0].boxes:
            class_id = int(box.cls[0])
            class_name = yolo_model.names[class_id]
            conf = float(box.conf[0])

            if class_name == 'no_helmet':
                no_helm += 1
                helmet_violation_counter += 1
                violation_text = "Motorcycle - No Helmet"
                
                log_to_csv(
                    VIOLATION_LOG, 
                    [datetime.now().strftime("%Y-%m-%d %H:%M:%S"), violation_text, f"{conf:.2f}", f"Video t={round(frame_idx/fps,1)}s"], 
                    ['DateTime', 'Type', 'Confidence', 'Source']
                )

            elif class_name == 'no_parking': # Ready for future expansion
                violation_text = "Illegal Parking Detected"
                
                log_to_csv(
                    VIOLATION_LOG, 
                    [datetime.now().strftime("%Y-%m-%d %H:%M:%S"), violation_text, f"{conf:.2f}", f"Video t={round(frame_idx/fps,1)}s"], 
                    ['DateTime', 'Type', 'Confidence', 'Source']
                )

        # ── Density overlay on frame ───────────────────────────────────────
        density = traffic_density(len(vehicles))
        cv2.putText(annotated, density, (10, H-15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)

        # ── Accident check ─────────────────────────────────────────────────
        if run_accident and frame_idx % int(acc_every_n) == 0:
            acc_label, acc_conf, acc_flag, acc_vehicles = \
                predict_accident_smoothed(frame, results)
            if acc_flag:
                accident_counter += 1
                t = round(frame_idx/fps, 2)
                accident_events.append(
                    {'frame':frame_idx,'time':t,'conf':round(acc_conf,3)})
                log_to_csv(ACCIDENT_LOG,
                    [datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                     frame_idx, t, round(acc_conf,3)],
                    ['DateTime','Frame','Video_Time_sec','Confidence'])

        if run_accident:
            annotated = draw_accident_overlay(
                annotated, acc_label, acc_conf, acc_flag, acc_vehicles)

        writer.write(annotated)

    cap.release()
    writer.release()

    # ── Stats summary ──────────────────────────────────────────────────────
    duration = round(total_frames/fps, 1)
    lines = [
        "="*42, "    📹 VIDEO ANALYSIS SUMMARY", "="*42,
        f"  Duration    : {duration}s ({total_frames} frames)",
        f"  FPS         : {fps:.0f}  |  Resolution: {W}x{H}", "",
        "─ Detection Totals ──────────────────",
        f"  🚗 Vehicles  : {total_v}",
        f"  🚦 Signs      : {total_s}",
        f"  🪖 Helmets   : {total_h}",
        f"  ⚠️  No-helmet : {no_helm}", "",
        "─ Accident Events ───────────────────",
    ]
    if accident_events:
        lines.append(f"  🚨 {len(accident_events)} accident(s) detected!")
        for ev in accident_events[:10]:
            lines.append(
                f"     t={ev['time']}s  frame={ev['frame']}  conf={ev['conf']:.0%}")
    else:
        lines.append("  ✅ No accidents detected")
    lines.append("="*42)

    acc_df   = get_log_df(ACCIDENT_LOG,
                    ['DateTime','Frame','Video_Time_sec','Confidence'])
    acc_file = ACCIDENT_LOG if os.path.exists(ACCIDENT_LOG) else None
    return out_path, "\n".join(lines), acc_df, acc_file

print("✅ detect_video ready!")

✅ detect_video ready!


In [29]:
def webcam_detect(pil_image, conf_threshold):
    if pil_image is None:
        return None, "⚠️ No frame captured."

    frame     = cv2.cvtColor(
        np.array(pil_image.convert("RGB")), cv2.COLOR_RGB2BGR)

    # ── Single YOLO model ─────────────────────────────────────────────────
    results   = yolo_model.predict(
        frame, conf=conf_threshold, verbose=False)
    annotated = results[0].plot(line_width=2, font_size=7)

    vehicles, signs, helmets = parse_detections(results)
    no_helmets = [h for h in helmets if h[0] == 'no_helmet']
    density    = traffic_density(len(vehicles))

    stats = (f"🚗 Vehicles: {len(vehicles)}  "
             f"🪖 Helmets: {len(helmets)}  "
             f"⚠️ No-helmet: {len(no_helmets)}  "
             f"🚦 Signs: {len(signs)}  "
             f"{density}")

    out_pil = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    return out_pil, stats

print("✅ webcam_detect ready!")


✅ webcam_detect ready!


In [30]:
def refresh_dashboard():
    vlog  = get_log_df(VIOLATION_LOG,
                ['DateTime','Type','Confidence','Source'])
    alog  = get_log_df(ACCIDENT_LOG,
                ['DateTime','Frame','Video_Time_sec','Confidence'])
    msg   = (f"Session Totals:\n"
             f"  🚗 Vehicles   : {vehicle_counter}\n"
             f"  🚨 Violations : {helmet_violation_counter}\n"
             f"  🚦 Signs      : {traffic_sign_counter}\n"
             f"  💥 Accidents  : {accident_counter}")
    vpath = VIOLATION_LOG if os.path.exists(VIOLATION_LOG) else None
    return msg, vlog, alog, vpath

def reset_counters():
    global vehicle_counter, helmet_violation_counter
    global traffic_sign_counter, accident_counter
    vehicle_counter          = 0
    helmet_violation_counter = 0
    traffic_sign_counter     = 0
    accident_counter         = 0
    ACCIDENT_HISTORY.clear()
    return "✅ All counters reset to zero!"

print("✅ refresh_dashboard + reset_counters ready!")


✅ refresh_dashboard + reset_counters ready!


In [35]:
# ── CELL 9: All Backend Functions (BULLETPROOF EDITION) ────────────────────
import os, shutil, yaml, random, csv
from pathlib import Path
from datetime import datetime
import pandas as pd
from ultralytics import YOLO

# ── CONFIG ─────────────────────────────────────────────────────────────────
CL_DIR    = os.path.join(os.getcwd(), 'continual')
CL_YAML   = os.path.join(os.getcwd(), 'cl_new.yaml')
DATA_YAML = r"C:\Users\ASUS\task3.yaml"
# Always start training from the original base model to prevent feedback loops
YOLO_PATH = r"C:\Users\ASUS\indian_traffic_model\task3_phase2\weights\best.pt"

VIOLATION_LOG = os.path.join(os.getcwd(), 'violation_logs.csv')
ACCIDENT_LOG  = os.path.join(os.getcwd(), 'accident_logs.csv')

OLD_TRAIN_IMG = r"C:\Users\ASUS\task3_dataset\images\train"
OLD_TRAIN_LBL = r"C:\Users\ASUS\task3_dataset\labels\train"
OLD_VAL_IMG   = r"C:\Users\ASUS\task3_dataset\images\valid"
OLD_VAL_LBL   = r"C:\Users\ASUS\task3_dataset\labels\valid"

# ── CSV helpers ────────────────────────────────────────────────────────────
def log_to_csv(path, row, headers):
    file_exists = os.path.isfile(path)
    with open(path, mode='a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if not file_exists:
            w.writerow(headers)
        w.writerow(row)

def get_log_df(path, cols):
    if os.path.exists(path):
        return pd.read_csv(path).tail(15).iloc[::-1]
    return pd.DataFrame(columns=cols)

# ── Counter reset ──────────────────────────────────────────────────────────
def reset_counters():
    global vehicle_counter, helmet_violation_counter
    global traffic_sign_counter, accident_counter
    vehicle_counter          = 0
    helmet_violation_counter = 0
    traffic_sign_counter     = 0
    accident_counter         = 0
    
    if 'ACCIDENT_HISTORY' in globals():
        globals()['ACCIDENT_HISTORY'].clear()
        
    return "✅ All counters reset to zero!"

def refresh_dashboard():
    vlog  = get_log_df(VIOLATION_LOG, ['DateTime','Type','Confidence','Source'])
    alog  = get_log_df(ACCIDENT_LOG, ['DateTime','Frame','Video_Time_sec','Confidence'])
    
    # 🚨 UI STRING UPDATED HERE
    msg   = (f"Session Totals:\n"
             f"  🚗 Vehicles              : {vehicle_counter}\n"
             f"  🚨 No-Helmet Violations  : {helmet_violation_counter}\n"
             f"  🚦 Signs                 : {traffic_sign_counter}\n"
             f"  💥 Accidents             : {accident_counter}")
    vpath = VIOLATION_LOG if os.path.exists(VIOLATION_LOG) else None
    return msg, vlog, alog, vpath

# ── YAML helpers ───────────────────────────────────────────────────────────
def update_yaml(new_class):
    with open(DATA_YAML, 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', [])
    
    if isinstance(names, dict):
        if new_class not in names.values():
            names[len(names)] = new_class
    else:
        if new_class not in names:
            names.append(new_class)
            
    data['names'] = names
    data['nc']    = len(names)
    with open(DATA_YAML, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)
    return f"✅ Class '{new_class}' added. Total classes: {len(names)}"

def reset_yaml_to_75():
    ORIGINAL_75_CLASSES = [
        "car", "motorcycle", "auto", "bus", "truck", "bicycle", "person", "animal", "cart",
        "no_entry", "no_parking", "no_stopping", "no_heavy_vehicles", "all_motor_vehicle_prohibited", 
        "cycle_prohibited", "pedestrian_prohibited", "left_turn_prohibited", "right_turn_prohibited", 
        "u_turn_prohibited", "straight_prohibited", "overtaking_prohibited", "truck_prohibited", 
        "horn_prohibited", "stop", "give_way", "compulsary_ahead", "compulsary_keep_left",
        "compulsary_keep_right", "compulsary_turn_left", "compulsary_turn_right", "compulsary_sound_horn",
        "roundabout", "pass_either_side", "speed_limit", "speed_limit_15", "speed_limit_20", 
        "speed_limit_30", "speed_limit_40", "speed_limit_50", "speed_limit_60", "speed_limit_70", 
        "speed_limit_80", "speed_limit_100", "speed_limit_120", "pedestrian_crossing",
        "school_ahead", "men_at_work", "traffic_signal_ahead", "narrow_bridge", "narrow_road", 
        "slippery_road", "steep_ascent", "steep_descent", "hump_or_rough_road", "falling_rocks", 
        "loose_gravel", "cattle_ahead", "cross_road", "t_intersection", "y_intersection",
        "side_road", "guarded_level_crossing", "unguarded_level_crossing", "dangerous_dip",
        "left_hand_curve", "right_hand_curve", "left_reverse_bend", "right_reverse_bend",
        "barrier_ahead", "major_road_ahead", "road_widens_ahead", "gap_in_median", "cycle_crossing",
        "helmet", "no_helmet"
    ]
    
    with open(DATA_YAML, 'w') as f:
        yaml.dump({
            'path' : r"C:\Users\ASUS\task3_dataset",
            'train': 'images/train',
            'val'  : 'images/valid',
            'test' : 'images/test',
            'nc'   : 75,
            'names': ORIGINAL_75_CLASSES,
        }, f, default_flow_style=False)
        
    for folder in ['continual', 'cl_mixed', 'cl_runs', 'cl_new.yaml']:
        path = os.path.join(os.getcwd(), folder)
        if os.path.exists(path):
            shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
            
    return "🗑️ SYSTEM RESET: YAML restored to 75 classes and CL folders wiped. Ready for fresh upload."

# ── Upload dataset ─────────────────────────────────────────────────────────
def upload_dataset(images, labels, class_name):
    if not images or not class_name.strip():
        return "⚠️ Please provide images and a class name."

    img_dir = os.path.join(CL_DIR, 'images')
    lbl_dir = os.path.join(CL_DIR, 'labels')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    img_map = {Path(img.name).stem: img for img in images}
    lbl_map = {Path(lbl.name).stem: lbl for lbl in labels} if labels else {}

    matched = []
    unmatched = []
    for stem, img_file in img_map.items():
        if stem in lbl_map:
            matched.append((stem, img_file, lbl_map[stem]))
        else:
            unmatched.append(stem)

    for stem, img_file, lbl_file in matched:
        shutil.copy(img_file.name, os.path.join(img_dir, os.path.basename(img_file.name)))
        shutil.copy(lbl_file.name, os.path.join(lbl_dir, os.path.basename(lbl_file.name)))

    yaml_msg = update_yaml(class_name.strip())
    
    with open(DATA_YAML, 'r') as f:
        full_yaml = yaml.safe_load(f)

    with open(CL_YAML, 'w') as f:
        yaml.dump({
            'path' : CL_DIR,
            'train': 'images',
            'val'  : 'images',
            'nc'   : full_yaml['nc'],
            'names': full_yaml['names']
        }, f, default_flow_style=False)

    return f"✅ Upload complete: {len(matched)} valid pairs saved for '{class_name}'.\n{yaml_msg}"

# ── Reload Model Function ──────────────────────────────────────────────
def reload_model():
    global yolo_model
    
    new_model_path = os.path.join(os.getcwd(), 'cl_runs', 'incremental', 'weights', 'best.pt')
    
    if not os.path.exists(new_model_path):
        return (
            "❌ New model not found!\n"
            "   Run 'Incremental Train' first.\n"
            f"   Expected: {new_model_path}"
        )

    try:
        yolo_model = YOLO(new_model_path)
        n_classes  = len(yolo_model.names)

        with open(DATA_YAML, 'r') as f:
            yaml_data = yaml.safe_load(f)
            
        nc    = yaml_data.get('nc', '?')
        names = yaml_data.get('names', [])

        lines = [
            "✅ Model reloaded successfully!",
            "",
            f"📋 New Model Info:",
            f"   Classes loaded : {n_classes}",
            f"   nc in YAML     : {nc}",
            "",
            "📊 Class List (last 5):",
        ]
        
        for i, name in enumerate(names[-5:]):
            idx = len(names) - 5 + i
            marker = " ← NEW" if idx >= 75 else ""
            lines.append(f"   [{idx}] {name}{marker}")

        lines += [
            "",
            "✅ Dashboard is now using the updated model!",
            "   Go to the Image Detection tab to test it."
        ]
        return "\n".join(lines)

    except Exception as e:
        return f"❌ Failed to reload model:\n{str(e)}"

# ── Incremental Training ───────────────────────────────────────────────────
def incremental_train():
    if not os.path.exists(CL_YAML):
        return "❌ No dataset found. Upload images + labels first."

    img_dir   = os.path.join(CL_DIR, 'images')
    img_count = len(list(Path(img_dir).glob('*.jpg'))) + \
                len(list(Path(img_dir).glob('*.png')))

    with open(CL_YAML, 'r') as f:
        cl_yaml_data = yaml.safe_load(f)

    names_dict = cl_yaml_data.get('names', {})
    if isinstance(names_dict, list):
        new_class_name = names_dict[-1]
    else:
        new_class_name = names_dict[max(names_dict.keys())]

    with open(DATA_YAML, 'r') as f:
        all_classes = yaml.safe_load(f).get('names', [])
    if isinstance(all_classes, dict):
        all_classes = list(all_classes.values())
    nc = len(all_classes)

    SIMILARITY_MAP = {
        'ambulance'  : ['car', 'truck', 'bus'], 
        'fire_truck' : ['truck', 'car', 'bus'],
        'police_car' : ['car'],
    }
    
    confuse_raw     = SIMILARITY_MAP.get(new_class_name.lower().replace(' ','_'), None)
    confuse_classes = confuse_raw if isinstance(confuse_raw, list) else (['car'] if 'car' in all_classes else [])
    confuse_ids     = [all_classes.index(cc) for cc in confuse_classes if cc in all_classes]

    confuse_limit = max(100, img_count // 4)
    confuse_count = {cid: 0 for cid in confuse_ids}

    mixed_dir = os.path.join(os.getcwd(), 'cl_mixed')
    if os.path.exists(mixed_dir):
        shutil.rmtree(mixed_dir)
    for sub in ['images/train','images/val', 'labels/train','labels/val']:
        os.makedirs(os.path.join(mixed_dir, sub), exist_ok=True)

    new_lbl_dir = os.path.join(CL_DIR, 'labels')
    new_copied  = 0
    new_imgs    = list(Path(img_dir).glob('*.jpg')) + list(Path(img_dir).glob('*.png'))

    for img in new_imgs:
        if img.suffix.lower() not in ['.jpg','.jpeg','.png']:
            continue
        lbl = Path(new_lbl_dir) / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy2(img, os.path.join(mixed_dir,'images/train', img.name))
            shutil.copy2(lbl, os.path.join(mixed_dir,'labels/train', lbl.name))
            new_copied += 1

    old_imgs = (list(Path(OLD_TRAIN_IMG).glob('*.jpg')) + list(Path(OLD_TRAIN_IMG).glob('*.png')))
    random.shuffle(old_imgs)

    replay_limit  = max(200, new_copied * 2) 
    replay_copied = 0

    for img in old_imgs:
        lbl = Path(OLD_TRAIN_LBL) / (img.stem + '.txt')
        if not lbl.exists():
            continue
        with open(lbl, 'r') as f:
            label_ids = [int(line.strip().split()[0]) for line in f if line.strip()]
            
        skip = False
        for cid in confuse_ids:
            if all(lid == cid for lid in label_ids):
                if confuse_count[cid] >= confuse_limit:
                    skip = True
                    break
                confuse_count[cid] += 1
        if skip:
            continue
            
        shutil.copy2(img, os.path.join(mixed_dir,'images/train', f"replay_{img.name}"))
        shutil.copy2(lbl, os.path.join(mixed_dir,'labels/train', f"replay_{img.stem}.txt"))
        replay_copied += 1
        if replay_copied >= replay_limit:
            break

    val_copied = 0
    if os.path.exists(OLD_VAL_IMG):
        val_imgs = (list(Path(OLD_VAL_IMG).glob('*.jpg')) + list(Path(OLD_VAL_IMG).glob('*.png')))
        random.shuffle(val_imgs)
        for img in val_imgs[:200]:
            lbl = Path(OLD_VAL_LBL) / (img.stem + '.txt')
            if lbl.exists():
                shutil.copy2(img, os.path.join(mixed_dir,'images/val', f"val_{img.name}"))
                shutil.copy2(lbl, os.path.join(mixed_dir,'labels/val', f"val_{img.stem}.txt"))
                val_copied += 1

    for img in new_imgs[:min(50, len(new_imgs))]:
        lbl = Path(new_lbl_dir) / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy2(img, os.path.join(mixed_dir,'images/val', f"newval_{img.name}"))
            shutil.copy2(lbl, os.path.join(mixed_dir,'labels/val', f"newval_{lbl.name}"))
            val_copied += 1

    mixed_yaml_path = os.path.join(mixed_dir, 'mixed.yaml')
    with open(mixed_yaml_path, 'w') as f:
        yaml.dump({
            'path' : mixed_dir,
            'train': 'images/train',
            'val'  : 'images/val',
            'nc'   : nc,
            'names': all_classes,
        }, f, default_flow_style=False)

    if img_count < 50:     epochs = 50
    elif img_count < 200:  epochs = 30
    elif img_count < 500:  epochs = 20
    elif img_count < 1000: epochs = 15
    else:                  epochs = 10

    try:
        model = YOLO(YOLO_PATH)
        model.train(
            data          = mixed_yaml_path,
            epochs        = epochs,
            imgsz         = 640,
            batch         = 8,
            freeze        = 10,       
            lr0           = 0.0001,   
            lrf           = 0.01,
            cos_lr        = True,
            weight_decay  = 0.001,    
            warmup_epochs = 3,
            workers       = 2,
            device        = 0,
            mosaic        = 1.0,
            fliplr        = 0.5,
            hsv_h         = 0.015,
            hsv_s         = 0.7,
            hsv_v         = 0.4,
            scale         = 0.5,
            translate     = 0.1,
            degrees       = 5.0,
            project       = os.path.join(os.getcwd(), 'cl_runs'),
            name          = 'incremental',
            exist_ok      = True,
        )

        confuse_msg = ""
        if confuse_ids:
            confuse_msg = "  Confuse classes limited:\n"
            for cc, cid in zip(confuse_classes, confuse_ids):
                confuse_msg += f"    '{cc}': {confuse_count[cid]}/{confuse_limit}\n"
        else:
            confuse_msg = "  No similar class limiting applied\n"

        return (
            f"✅ Incremental training complete!\n\n"
            f"📊 Dataset Summary:\n"
            f"   New class        : {new_class_name}\n"
            f"   New images       : {new_copied}\n"
            f"   Replay images    : {replay_copied}\n"
            f"   Replay ratio     : 1:{replay_limit//max(new_copied,1)}\n"
            f"{confuse_msg}"
            f"   Val images       : {val_copied}\n"
            f"   Total classes    : {nc}\n"
            f"   Epochs           : {epochs}\n\n"
            f"⚙️  Key Fixes Applied:\n"
            f"   confuse_limit = img//4 = {confuse_limit} ✅\n"
            f"   replay_ratio  = 2×     = {replay_limit} ✅\n"
            f"   freeze=10 (LOCKED BACKBONE)             ✅\n"
            f"   augmentation ON                         ✅\n\n"
            f"   → Click 'Reload New Model'"
        )
    except Exception as e:
        return f"❌ Training failed:\n{str(e)}"

print("✅ Cell 9 Ready: Bulletproof Continual Learning initialized.")

✅ Cell 9 Ready: Bulletproof Continual Learning initialized.


In [36]:
# ── CELL 10: Gradio Dashboard ─────────────────────────────────────────────
import gradio as gr

conf_slider     = lambda: gr.Slider(0.10, 0.80, value=0.25, step=0.05,
                    label="Detection Confidence Threshold")
accident_toggle = lambda: gr.Checkbox(value=True,
                    label="Enable Accident Detection (CNN)")

theme = gr.themes.Soft(
    primary_hue="blue", secondary_hue="red", neutral_hue="slate",
).set(
    body_background_fill="#0f1117",
    body_text_color="#f0f0f0",
    block_background_fill="#1a1d2e",
    block_border_color="#2e3250",
)

with gr.Blocks(title="🚦 Indian Traffic AI Dashboard") as demo:

    gr.HTML("""
    <div style="text-align:center; padding:20px 0 10px 0;">
        <h1 style="font-size:2.2em; color:#60a5fa; margin:0;">
            🚦 Indian Traffic AI — Smart City Dashboard
        </h1>
        <p style="color:#94a3b8; font-size:1.05em; margin-top:6px;">
            Vehicle · Signs · Helmet Violation · Accident · Density · Continual Learning
        </p>
        <p style="color:#475569; font-size:0.85em;">
            YOLO (75 classes) + CNN Accident Detector · MIT Campus Anna University
        </p>
    </div>""")

    with gr.Row():
        gr.Label(value="YOLO — 75 classes",  label="🤖 Detection Model")
        gr.Label(value="Vehicles (0–8)",      label="🚗 Vehicles")
        gr.Label(value="Signs (9–72)",        label="🚦 Signs")
        gr.Label(value="Helmet (73–74)",      label="🪖 Helmet")

    gr.HTML("<hr style='border-color:#2e3250; margin:10px 0;'>")

    with gr.Tab("🖼️ Image Detection"):
        gr.Markdown("### Upload any Indian road image for full analysis")
        with gr.Row():
            with gr.Column(scale=1):
                img_input    = gr.Image(sources=["upload"], type="pil",
                                label="📷 Input Image", height=380)
                img_conf     = conf_slider()
                img_accident = accident_toggle()
                img_btn      = gr.Button("🔍 Detect", variant="primary", size="lg")
            with gr.Column(scale=1):
                img_output = gr.Image(label="✅ Annotated Output", height=380)
                img_stats  = gr.Textbox(label="📊 Detection Stats", lines=22)
                img_vlog   = gr.Dataframe(label="🚨 Violation Log", interactive=False)
        img_btn.click(
            fn=detect_image,
            inputs=[img_input, img_conf, img_accident],
            outputs=[img_output, img_stats, img_vlog])

    with gr.Tab("🎥 Video Detection"):
        gr.Markdown("### Upload a road video — annotated output + full report")
        with gr.Row():
            with gr.Column(scale=1):
                vid_input    = gr.Video(label="📹 Upload Video")
                vid_conf     = conf_slider()
                vid_accident = accident_toggle()
                vid_acc_n    = gr.Slider(1, 15, value=5, step=1,
                                label="Run Accident Check Every N Frames")
                vid_btn      = gr.Button("▶️ Process Video", variant="primary", size="lg")
            with gr.Column(scale=1):
                vid_output = gr.Video(label="✅ Processed Output Video")
                vid_stats  = gr.Textbox(label="📊 Video Analysis Report", lines=12)
                gr.Markdown("### 🗄️ Accident Log")
                log_df     = gr.Dataframe(label="Recent Accidents", interactive=False)
                log_file   = gr.File(label="📥 Download Accident CSV")
        vid_btn.click(
            fn=detect_video,
            inputs=[vid_input, vid_conf, vid_accident, vid_acc_n],
            outputs=[vid_output, vid_stats, log_df, log_file])

    with gr.Tab("📷 Webcam Detection"):
        gr.Markdown("### Capture from webcam or upload a snapshot")
        with gr.Row():
            with gr.Column(scale=1):
                cam_input = gr.Image(sources=["webcam","upload"],
                              type="pil", label="📷 Webcam / Upload", height=360)
                cam_conf  = conf_slider()
                cam_btn   = gr.Button("📸 Snap & Detect", variant="primary", size="lg")
            with gr.Column(scale=1):
                cam_output = gr.Image(label="✅ Detection Result", height=360)
                cam_stats  = gr.Textbox(label="📊 Detection Stats", lines=6)
        cam_btn.click(
            fn=webcam_detect,
            inputs=[cam_input, cam_conf],
            outputs=[cam_output, cam_stats])

    with gr.Tab("📊 Violation Dashboard"):
        gr.Markdown("### Live session counters + full violation & accident log")
        with gr.Row():
            with gr.Column(scale=1):
                dash_refresh = gr.Button("🔄 Refresh Dashboard", variant="primary")
                dash_reset   = gr.Button("🗑️ Reset Counters",   variant="stop")
                dash_status  = gr.Textbox(label="Session Totals", lines=6)
            with gr.Column(scale=1):
                dash_vlog = gr.Dataframe(label="🚨 Violation Log (newest first)",
                                         interactive=False)
                dash_alog = gr.Dataframe(label="💥 Accident Log",
                                         interactive=False)
                dash_file = gr.File(label="📥 Download Violation CSV")
        dash_refresh.click(fn=refresh_dashboard,
            outputs=[dash_status, dash_vlog, dash_alog, dash_file])
        dash_reset.click(fn=reset_counters,
            outputs=[dash_status])

    with gr.Tab("🧠 Continual Learning"):
        gr.Markdown("### Upload new class images + labels and retrain incrementally")
        with gr.Row():
            with gr.Column(scale=1):
                # 1. Textbox and Action Buttons at the TOP
                cl_classname  = gr.Textbox(
                                 label="New Class Name",
                                 placeholder="e.g. e-scooter, ambulance...")
                
                with gr.Row():
                    cl_upload_btn = gr.Button("📤 Upload Dataset",    variant="primary")
                    cl_train_btn  = gr.Button("🚀 Incremental Train", variant="secondary")
                
                with gr.Row():
                    cl_reload_btn = gr.Button("🔄 Reload New Model",  variant="primary")
                    cl_reset_btn  = gr.Button("🗑️ Reset to 75 Classes", variant="stop")
                
                gr.HTML("<hr style='border-color:#2e3250; margin:15px 0;'>")
                
                # 2. File Uploaders at the BOTTOM
                cl_images     = gr.File(file_count="multiple",
                                 label="📁 1. Upload Images Here")
                cl_labels     = gr.File(file_count="multiple",
                                 label="📄 2. Upload Labels Here (.txt YOLO format)")
                
            with gr.Column(scale=1):
                cl_status = gr.Textbox(label="📋 Status Log", lines=18)
                gr.Markdown("""
**Step by Step:**

**Step 1 — Upload**
- Type new class name above
- Drop Images & Labels below
- Click **Upload Dataset**
- YAML auto-updated (nc+1) ✅

**Step 2 — Train**
- Click **Incremental Train**
- Backbone frozen → old classes safe
- Epochs auto-scale based on image count
- New model saved to `cl_runs/`

**Step 3 — Reload**
- Click **Reload New Model**
- Dashboard instantly uses new model
- No restart needed ✅
- New class now detectable!

**This is Class-Incremental Continual Learning** ✅
                """)
                
        # ── Click Events ──
        cl_upload_btn.click(
            fn=upload_dataset,
            inputs=[cl_images, cl_labels, cl_classname],
            outputs=[cl_status])
        cl_train_btn.click(
            fn=incremental_train,
            outputs=[cl_status])
        cl_reload_btn.click(
            fn=reload_model,
            outputs=[cl_status])
        cl_reset_btn.click(
            fn=reset_yaml_to_75,
            outputs=[cl_status])

    with gr.Tab("ℹ️ Model Info"):
        gr.Markdown("""
## 🤖 Model Architecture

| Component | Details |
|-----------|---------|
| **CL Model** | YOLOv8s — 75 classes (vehicles + signs + helmet) |
| **Accident CNN** | Binary CNN — accident / no_accident (250×250) |
| **mAP@0.5** | 85.5% overall · Helmet 92.5% · Signs 99.5% |

## 📊 Traffic Density Thresholds
| Vehicles in Frame | Level |
|---|---|
| < 5 | 🟢 Low Traffic |
| 5–14 | 🟡 Medium Traffic |
| 15+ | 🔴 Heavy Traffic |

## 🚨 Violation Detection Logic
- Single YOLO model detects motorcycles + helmet/no_helmet together
- No-helmet on motorcycle → violation flagged + logged to CSV
- Violation logged with timestamp

## 🧠 Accident Detection Logic
- CNN runs every N frames (configurable slider)
- Temporal smoothing — needs 3 of last 5 frames to trigger
- Dynamic threshold based on vehicle count in frame
- Minimum 2 vehicles required before alert fires

## 🔄 Continual Learning Flow
- Upload new class images + YOLO labels
- YAML auto-updated (nc: 75 → 76)
- Backbone frozen during training
- Click Reload — no restart needed

## 📍 MIT Campus, Anna University
        """)

    gr.HTML("""
    <div style="text-align:center; color:#475569; font-size:0.82em; padding:15px 0 5px 0;">
        🚦 Indian Traffic AI Dashboard · MIT Campus Anna University
    </div>""")

demo.launch(theme=theme, share=True, show_error=True)

* Running on local URL:  http://127.0.0.1:7865

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/03/31 20:50:34 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout


In [11]:
# ── Check Current Class List ───────────────────────────────────────────────
import yaml

DATA_YAML = r"C:\Users\ASUS\task3.yaml"

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

names = data.get('names', [])
nc    = data.get('nc', 0)

print("="*45)
print("  📋 CURRENT CLASS LIST")
print("="*45)
print(f"  Total classes (nc) : {nc}")
print(f"  Names count        : {len(names)}")
print("="*45)

print("\n  🚗 Vehicles (0–8):")
for i in range(min(9, len(names))):
    print(f"     [{i}] {names[i]}")

print("\n  🚦 Signs (9–72):")
for i in range(9, min(73, len(names))):
    print(f"     [{i}] {names[i]}")

print("\n  🪖 Helmet (73–74):")
for i in range(73, min(75, len(names))):
    print(f"     [{i}] {names[i]}")

if len(names) > 75:
    print("\n  🆕 New Classes Added via CL:")
    for i in range(75, len(names)):
        print(f"     [{i}] {names[i]}  ← NEW")

print("\n" + "="*45)
print(f"  nc match  : {'✅ OK' if nc == len(names) else '❌ MISMATCH — run update_yaml()'}")
print(f"  Original  : 75 classes")
print(f"  Added     : {max(0, len(names)-75)} new classes")
print("="*45)


  📋 CURRENT CLASS LIST
  Total classes (nc) : 76
  Names count        : 76

  🚗 Vehicles (0–8):
     [0] car
     [1] motorcycle
     [2] auto
     [3] bus
     [4] truck
     [5] bicycle
     [6] person
     [7] animal
     [8] cart

  🚦 Signs (9–72):
     [9] no_entry
     [10] no_parking
     [11] no_stopping
     [12] no_heavy_vehicles
     [13] all_motor_vehicle_prohibited
     [14] cycle_prohibited
     [15] pedestrian_prohibited
     [16] left_turn_prohibited
     [17] right_turn_prohibited
     [18] u_turn_prohibited
     [19] straight_prohibited
     [20] overtaking_prohibited
     [21] truck_prohibited
     [22] horn_prohibited
     [23] stop
     [24] give_way
     [25] compulsary_ahead
     [26] compulsary_keep_left
     [27] compulsary_keep_right
     [28] compulsary_turn_left
     [29] compulsary_turn_right
     [30] compulsary_sound_horn
     [31] roundabout
     [32] pass_either_side
     [33] speed_limit
     [34] speed_limit_15
     [35] speed_limit_20
     [36] speed

ERROR:asyncio:Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
ERROR:asyncio:Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\proac

In [18]:
print(list(yolo_model.names.values()))

['car', 'motorcycle', 'auto', 'bus', 'truck', 'bicycle', 'person', 'animal', 'cart', 'no_entry', 'no_parking', 'no_stopping', 'no_heavy_vehicles', 'all_motor_vehicle_prohibited', 'cycle_prohibited', 'pedestrian_prohibited', 'left_turn_prohibited', 'right_turn_prohibited', 'u_turn_prohibited', 'straight_prohibited', 'overtaking_prohibited', 'truck_prohibited', 'horn_prohibited', 'stop', 'give_way', 'compulsary_ahead', 'compulsary_keep_left', 'compulsary_keep_right', 'compulsary_turn_left', 'compulsary_turn_right', 'compulsary_sound_horn', 'roundabout', 'pass_either_side', 'speed_limit', 'speed_limit_15', 'speed_limit_20', 'speed_limit_30', 'speed_limit_40', 'speed_limit_50', 'speed_limit_60', 'speed_limit_70', 'speed_limit_80', 'speed_limit_100', 'speed_limit_120', 'pedestrian_crossing', 'school_ahead', 'men_at_work', 'traffic_signal_ahead', 'narrow_bridge', 'narrow_road', 'slippery_road', 'steep_ascent', 'steep_descent', 'hump_or_rough_road', 'falling_rocks', 'loose_gravel', 'cattle_ah

ERROR:asyncio:Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host
ERROR:asyncio:Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
  File "C:\Users\ASUS\anaconda3\envs\python-gpu\Lib\asyncio\proac

In [30]:
import os
from glob import glob

# ⚠️ CHANGE THIS to the exact folder where your Roboflow .txt labels are!
ROBOFLOW_LABELS_DIR = r"C:\Users\ASUS\Downloads\ambulance.yolov8 (1)\train\labels"

# In our system, the 76th class gets ID 75 (since we start at 0)
NEW_CLASS_ID = 75

txt_files = glob(os.path.join(ROBOFLOW_LABELS_DIR, "*.txt"))
fixed_count = 0

for txt_file in txt_files:
    with open(txt_file, "r") as f:
        lines = f.readlines()
        
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 5:
            # Change the Roboflow '0' to our system's '75'
            parts[0] = str(NEW_CLASS_ID) 
            new_lines.append(" ".join(parts) + "\n")
            
    with open(txt_file, "w") as f:
        f.writelines(new_lines)
    fixed_count += 1

print(f"✅ Successfully converted {fixed_count} label files to Class ID {NEW_CLASS_ID}!")

✅ Successfully converted 841 label files to Class ID 75!


In [35]:
# ── Reset yaml back to original 75 classes ────────────────────────────────
import yaml

DATA_YAML = r"C:\Users\ASUS\task3.yaml"

ORIGINAL_75_CLASSES = [
    # Vehicles (0-8)
    "car", "motorcycle", "auto", "bus", "truck",
    "bicycle", "person", "animal", "cart",
    # Signs (9-72)
    "no_entry", "no_parking", "no_stopping", "no_heavy_vehicles",
    "all_motor_vehicle_prohibited", "cycle_prohibited",
    "pedestrian_prohibited", "left_turn_prohibited",
    "right_turn_prohibited", "u_turn_prohibited", "straight_prohibited",
    "overtaking_prohibited", "truck_prohibited", "horn_prohibited",
    "stop", "give_way", "compulsary_ahead", "compulsary_keep_left",
    "compulsary_keep_right", "compulsary_turn_left",
    "compulsary_turn_right", "compulsary_sound_horn",
    "roundabout", "pass_either_side", "speed_limit",
    "speed_limit_15", "speed_limit_20", "speed_limit_30",
    "speed_limit_40", "speed_limit_50", "speed_limit_60",
    "speed_limit_70", "speed_limit_80", "speed_limit_100",
    "speed_limit_120", "pedestrian_crossing",
    "school_ahead", "men_at_work", "traffic_signal_ahead",
    "narrow_bridge", "narrow_road", "slippery_road",
    "steep_ascent", "steep_descent", "hump_or_rough_road",
    "falling_rocks", "loose_gravel", "cattle_ahead",
    "cross_road", "t_intersection", "y_intersection",
    "side_road", "guarded_level_crossing",
    "unguarded_level_crossing", "dangerous_dip",
    "left_hand_curve", "right_hand_curve",
    "left_reverse_bend", "right_reverse_bend",
    "barrier_ahead", "major_road_ahead",
    "road_widens_ahead", "gap_in_median", "cycle_crossing",
    # Helmet (73-74)
    "helmet", "no_helmet",
]

# Verify count
print(f"Class count to restore: {len(ORIGINAL_75_CLASSES)}")
assert len(ORIGINAL_75_CLASSES) == 75, "❌ Count mismatch!"

# Backup current yaml first
import shutil
shutil.copy(DATA_YAML, DATA_YAML.replace('.yaml','_backup_77.yaml'))
print(f"✅ Backup saved: task3_backup_77.yaml")

# Write clean 75 class yaml
with open(DATA_YAML, 'w') as f:
    yaml.dump({
        'path' : r"C:\Users\ASUS\task3_dataset",
        'train': 'images/train',
        'val'  : 'images/valid',
        'test' : 'images/test',
        'nc'   : 75,
        'names': ORIGINAL_75_CLASSES,
    }, f, default_flow_style=False)

print("✅ task3.yaml reset to original 75 classes!")
print(f"\n   Vehicles : [0–8]")
print(f"   Signs    : [9–72]")
print(f"   Helmet   : [73–74]")
print(f"   Total    : 75")
print(f"\n⚠️  Note: Your YOLO model (best.pt) still has 75 classes")
print(f"   → No model retraining needed")
print(f"   → YAML now matches model ✅")

Class count to restore: 75
✅ Backup saved: task3_backup_77.yaml
✅ task3.yaml reset to original 75 classes!

   Vehicles : [0–8]
   Signs    : [9–72]
   Helmet   : [73–74]
   Total    : 75

⚠️  Note: Your YOLO model (best.pt) still has 75 classes
   → No model retraining needed
   → YAML now matches model ✅


In [36]:
# ── Verify yaml is clean ───────────────────────────────────────────────────
import yaml

with open(r"C:\Users\ASUS\task3.yaml", 'r') as f:
    data = yaml.safe_load(f)

names = data.get('names', [])
nc    = data.get('nc', 0)

print("="*45)
print("  📋 YAML VERIFICATION")
print("="*45)
print(f"  nc          : {nc}")
print(f"  names count : {len(names)}")
print(f"  Match       : {'✅ OK' if nc == len(names) else '❌ MISMATCH'}")
print(f"  Correct     : {'✅ 75 classes' if len(names)==75 else '❌ Wrong count'}")
print("="*45)
print(f"\n  First 3 : {names[:3]}")
print(f"  Last  3 : {names[-3:]}")
print(f"\n  [73] {names[73]}")
print(f"  [74] {names[74]}")
if len(names) > 75:
    print(f"\n  ⚠️  Extra classes still present:")
    for i in range(75, len(names)):
        print(f"     [{i}] {names[i]}")
else:
    print(f"\n  ✅ No extra classes — clean 75 class yaml!")

  📋 YAML VERIFICATION
  nc          : 75
  names count : 75
  Match       : ✅ OK
  Correct     : ✅ 75 classes

  First 3 : ['car', 'motorcycle', 'auto']
  Last  3 : ['cycle_crossing', 'helmet', 'no_helmet']

  [73] helmet
  [74] no_helmet

  ✅ No extra classes — clean 75 class yaml!


In [37]:
# ── Clean all CL training folders ─────────────────────────────────────────
import shutil, os

folders = [
    os.path.join(os.getcwd(), 'continual'),
    os.path.join(os.getcwd(), 'cl_mixed'),
    os.path.join(os.getcwd(), 'cl_runs'),
    os.path.join(os.getcwd(), 'cl_new.yaml'),
]

for folder in folders:
    if os.path.exists(folder):
        if os.path.isdir(folder):
            shutil.rmtree(folder)
        else:
            os.remove(folder)
        print(f"✅ Removed: {folder}")
    else:
        print(f"⏭️  Not found: {folder}")

print("\n✅ All CL folders cleaned!")
print("   Ready for fresh fire_truck training")

✅ Removed: C:\Users\ASUS\continual
✅ Removed: C:\Users\ASUS\cl_mixed
✅ Removed: C:\Users\ASUS\cl_runs
✅ Removed: C:\Users\ASUS\cl_new.yaml

✅ All CL folders cleaned!
   Ready for fresh fire_truck training
